# Assembly-to-Order (ATO) Pizza Optimization Problem

## Problem Description
We are working on an Assembly-to-Order (ATO) optimization problem involving the production of two types of pizza: **Margherita** and **4 seasons**. We have one pizza maker available to assemble the pizzas, and we are interested in optimizing the pizza production while satisfying the constraints on ingredients and time.

### Ingredients
- **Dough**
- **Tomato Sauce**
- **Vegetables**

### Pizza Types
- **Margherita** (uses 1 unit of dough, 1 unit of tomato sauce, and no vegetables)
- **4 Seasons** (uses 1 unit of dough, 1 unit of tomato sauce, and 1 unit of vegetables)

### Costs and Time
- **Cost of Ingredients**: 
  - Dough: 1€
  - Tomato Sauce: 1€
  - Vegetables: 3€
- **Prices of Pizzas**:
  - Margherita: 6€
  - 4 Seasons: 8.5€
- **Time to Produce a Pizza**:
  - Margherita: 0.5 hours
  - 4 Seasons: 0.25 hours for each unit of dough, tomato sauce, and vegetables
- **Available time**: The pizza maker has a total of 6 hours.

### Constraints
- We have three main ingredients, and their availability is limited.
- We need to consider multiple demand scenarios for the pizzas.
- We do not care about the `y` variables, which represent recourse actions in this two-stage stochastic program. We are only interested in the decision variables `x` which represent the pizza amounts to produce.

### Objective
The objective of this optimization problem is to maximize the profit from pizza sales, considering the costs of ingredients, the price of each pizza, and the available resources (ingredients and time).

## Implementation


In this part, we define all the variables that were provided by the statement.

In [1]:

import numpy as np

components_cost = np.array([1, 1, 3]) # euros
num_components = len(components_cost)
pizza_selling_price = np.array([6, 8.5]) # euros
n_pizzas = len(pizza_selling_price)

# Machine
time_required_per_component = np.array([0.5, 0.25, 0.25]) # hours
machine_availability = 6 # hours

map_components_cost_to_pizza_cost = np.array([[1, 1, 0],
                                              [1, 1, 1]])
map_pizza_to_necessary_components = map_components_cost_to_pizza_cost.T

# possible scenarios [demmand_pizza1, demmand_pizza2]
scenario_0 = np.array([100, 400]) # pi = 0.3
scenario_1 = np.array([90, 100])  # pi = 0.1
scenario_2 = np.array([80, 250])  # pi = 0.6

demand = np.array([scenario_0, scenario_1, scenario_2])
prob = [0.3, 0.1, 0.6]
n_scenarios = len(demand)
scenarios = range(n_scenarios)


Here the code creates the model and set the variables we want to optimize.

In [2]:
import gurobipy as gb

pizza_model = gb.Model("pizza_restaurant")

# X: target var: number of components I should have
x_stock = pizza_model.addMVar(shape=num_components, lb=0, name="x") # column vector
# Y: stocastic var: number of pizzas I make in each scenario
pizza_sold_amount = pizza_model.addMVar(shape=(n_scenarios, n_pizzas), lb=0, name="y") # each line have the sold quantity


Restricted license - for non-production use only - expires 2025-11-24


Below, a function is made to abstract the calculation of the profit for a given scenario.

In [3]:

def calc_profit(scenario):
    return pizza_sold_amount[scenario] @ pizza_selling_price.T - components_cost @ x_stock

calc_profit(0)

<MLinExpr ()  >
array( 6.0 <gurobi.Var *Awaiting Model Update*> + 8.5 <gurobi.Var *Awaiting Model Update*> + -1.0 <gurobi.Var *Awaiting Model Update*> + -1.0 <gurobi.Var *Awaiting Model Update*> + -3.0 <gurobi.Var *Awaiting Model Update*>)

Now we can weight each profit by their probabilities. In this way we find the expected value of profit, that's the function that we want to maximize.

In [4]:
profit_s = [calc_profit(s) for s in scenarios]
gb.quicksum(prob[s] * profit_s[s] for s in scenarios)

<MLinExpr ()   >
array( 1.7999999999999998 <gurobi.Var *Awaiting Model Update*> + 2.55 <gurobi.Var *Awaiting Model Update*> + -0.3 <gurobi.Var *Awaiting Model Update*> + -0.3 <gurobi.Var *Awaiting Model Update*> + -0.8999999999999999 <gurobi.Var *Awaiting Model Update*> + 0.6000000000000001 <gurobi.Var *Awaiting Model Update*> + 0.8500000000000001 <gurobi.Var *Awaiting Model Update*> + -0.1 <gurobi.Var *Awaiting Model Update*> + -0.1 <gurobi.Var *Awaiting Model Update*> + -0.30000000000000004 <gurobi.Var *Awaiting Model Update*> + 3.5999999999999996 <gurobi.Var *Awaiting Model Update*> + 5.1 <gurobi.Var *Awaiting Model Update*> + -0.6 <gurobi.Var *Awaiting Model Update*> + -0.6 <gurobi.Var *Awaiting Model Update*> + -1.7999999999999998 <gurobi.Var *Awaiting Model Update*>)

The output above is our objective function.

In [5]:
from gurobipy import GRB

## Now we maximize the expected profit for each scenario ponderating each probability:
pizza_model.setObjective(
    gb.quicksum(prob[s] * profit_s[s] for s in scenarios),
    GRB.MAXIMIZE
)

Now we need to set the constraints of our problem. This part could be very confusing because of the matrix operations, so here is a recap about the format of the matrixes:

| Variable/Matrix                           | Shape                   | Description                                                       | Type                                |
|-------------------------------------------|--------------------------|-------------------------------------------------------------------|-------------------------------------|
| `x_stock`                                 | `(num_components, 1)`     | Decision variable: amount of each component to stock.             | Column vector                      |
| `pizza_sold_amount`                       | `(n_scenarios, n_pizzas)` | Stochastic variable: number of pizzas sold in each scenario.       | Matrix (each row is a scenario)     |
| `map_pizza_to_necessary_components`       | `(num_components, n_pizzas)` | Maps pizzas to the components they require.                       | Mapping matrix                     |
| `time_required_per_component`             | `(1, num_components)`     | Time required for each component.                                 | Row vector                         |
| `demand`                                  | `(n_scenarios, n_pizzas)` | Demand for pizzas in each scenario.                               | Matrix (each row is a scenario)     |


In [6]:

pizza_model.addConstr(pizza_sold_amount <= demand, name="demand_constrains")

pizza_model.addConstr(time_required_per_component @ x_stock <= machine_availability, name="machine_constrains")

for s in scenarios:
    pizza_model.addConstr(
        (map_pizza_to_necessary_components @ pizza_sold_amount[s]) <= x_stock, 
        name=f"components_constrains_scenario_{s}"
    )


Now we ask to gurobi to find the maximum of this function:

In [7]:

# Solve
pizza_model.optimize()

# Output results to analyse each scenario
if pizza_model.status == GRB.OPTIMAL:
    print(f"Optimal number of components to buy: {x_stock.X}")
    for s in scenarios:
        print(f"Scenario {s}: Sold {pizza_sold_amount[s].X} pizzas")

# Print out the model with all constraints and variables
pizza_model.write("pizza_restaurant_model.lp")


Gurobi Optimizer version 11.0.3 build v11.0.3rc0 (linux64 - "Ubuntu 24.04 LTS")

CPU model: Intel(R) Core(TM) i7-8700 CPU @ 3.20GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 6 physical cores, 12 logical processors, using up to 12 threads

Optimize a model with 16 rows, 9 columns and 33 nonzeros
Model fingerprint: 0x5bcd75e2
Coefficient statistics:
  Matrix range     [2e-01, 1e+00]
  Objective range  [6e-01, 5e+00]
  Bounds range     [0e+00, 0e+00]
  RHS range        [6e+00, 4e+02]
Presolve removed 6 rows and 0 columns
Presolve time: 0.01s
Presolved: 10 rows, 9 columns, 27 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    9.2500100e+01   3.003900e+01   0.000000e+00      0s
       4    3.2000000e+01   0.000000e+00   0.000000e+00      0s

Solved in 4 iterations and 0.01 seconds (0.00 work units)
Optimal objective  3.200000000e+01
Optimal number of components to buy: [8. 8. 0.]
Scenario 0: Sold [8. 0.] pizzas
Scenario 1: Sold [8. 0.] pizzas
Scenario

This result may seam wrong in a first view, because of the high demand. But you can see in the file `pizza_restaurant_model.lp` that the model is corect. What is limiting the amount of pizzas is the machine.

In [8]:

# Check if the constraints are binding (tight) or not
if pizza_model.Status == GRB.OPTIMAL:
    for constr in pizza_model.getConstrs():
        # Get LHS value of the constraint
        lhs_value = pizza_model.getRow(constr).getValue()
        # Get RHS value of the constraint
        rhs_value = constr.RHS
        # Calculate the slack (difference between LHS and RHS)
        slack = rhs_value - lhs_value

        if abs(slack) < 1e-6:  # Small tolerance to account for numerical precision
            print(f"Constraint {constr.ConstrName}:")
            print(f"  LHS = {lhs_value}, RHS = {rhs_value}, Slack = {slack}")
            print("  This constraint is binding.")
            

Constraint machine_constrains:
  LHS = 6.0, RHS = 6.0, Slack = 0.0
  This constraint is binding.
Constraint components_constrains_scenario_0[0]:
  LHS = 0.0, RHS = -0.0, Slack = -0.0
  This constraint is binding.
Constraint components_constrains_scenario_0[1]:
  LHS = 0.0, RHS = -0.0, Slack = -0.0
  This constraint is binding.
Constraint components_constrains_scenario_0[2]:
  LHS = 0.0, RHS = -0.0, Slack = -0.0
  This constraint is binding.
Constraint components_constrains_scenario_1[0]:
  LHS = 0.0, RHS = -0.0, Slack = -0.0
  This constraint is binding.
Constraint components_constrains_scenario_1[1]:
  LHS = 0.0, RHS = -0.0, Slack = -0.0
  This constraint is binding.
Constraint components_constrains_scenario_1[2]:
  LHS = 0.0, RHS = -0.0, Slack = -0.0
  This constraint is binding.
Constraint components_constrains_scenario_2[0]:
  LHS = 0.0, RHS = -0.0, Slack = -0.0
  This constraint is binding.
Constraint components_constrains_scenario_2[1]:
  LHS = 0.0, RHS = -0.0, Slack = -0.0
  Thi

How we can see in the output above, the model found the solution in the limit of the components output, what is expected because we dont wanto to buy more components than we need, and its limited by the machine constrains.